# Zero-Shot Text Classification with a Pipeline

In [1]:
from transformers import pipeline

# We'll use a model specialized in zero-shot classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:

sequence = "I absolutely loved this movie. The acting was great, and the storyline was captivating."
candidate_labels = ["positive", "negative", "neutral"]

result = classifier(sequence, candidate_labels)
print(result)


{'sequence': 'I absolutely loved this movie. The acting was great, and the storyline was captivating.', 'labels': ['positive', 'neutral', 'negative'], 'scores': [0.9519802331924438, 0.036022938787937164, 0.011996792629361153]}


# Few-Shot Prompting for Text Generation (in-context learning)

In [3]:
from transformers import pipeline

# Use a smaller model like DistilGPT2 instead of GPT-Neo
generator = pipeline("text-generation", model="distilgpt2")


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
prompt = """Summarize the following text:

Example 1:
Text: "Alice was beginning to get very tired of sitting by her sister on the bank, ...
Summary: "Alice grows bored and follows a rabbit into a bizarre world."

Example 2:
Text: "John found a mysterious map in his attic, which led him to hidden treasure. ...
Summary: "John discovers a hidden map and embarks on a treasure hunt."

Now your turn:
Text: "Scientists have discovered a new species of frog in the Amazon rainforest.
This frog has vibrant red markings, is nocturnal, and lives in the upper canopy..."
Summary:"""

output = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.7)
print(output[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize the following text:

Example 1:
Text: "Alice was beginning to get very tired of sitting by her sister on the bank, ...
Summary: "Alice grows bored and follows a rabbit into a bizarre world."

Example 2:
Text: "John found a mysterious map in his attic, which led him to hidden treasure. ...
Summary: "John discovers a hidden map and embarks on a treasure hunt."

Now your turn:
Text: "Scientists have discovered a new species of frog in the Amazon rainforest.
This frog has vibrant red markings, is nocturnal, and lives in the upper canopy..."
Summary: "John is studying how the frog can grow up in the rainforest. He has a great sense of humor and good manners."
Summary: "


# Chain of Thought Examples

# Example 1: Mathematical reasoning with step-by-step thinking


In [5]:
def solve_math_problem(generator, problem):
    # Using few-shot examples to guide chain of thought
    cot_prompt = f"""
    Question: If Alice has 15 candies and eats 5, then buys 8 more, how many candies does she have?
    Let's solve step by step:
    1. Start: Alice has 15 candies
    2. After eating: 15 - 5 = 10 candies
    3. After buying: 10 + 8 = 18 candies
    Answer: Alice has 18 candies.

    Question: If Bob has 24 stickers and gives 6 away, then finds 5 more, how many stickers does he have?
    Let's solve step by step:
    1. Start: Bob has 24 stickers
    2. After giving: 24 - 6 = 18 stickers
    3. After finding: 18 + 5 = 23 stickers
    Answer: Bob has 23 stickers.

    Question: {problem}
    Let's solve step by step:
    """

    return generator(cot_prompt, max_new_tokens=100, do_sample=True, temperature=0.7)

math_problem = "If Tom has 27 marbles and loses 5, then finds 13 more, how many marbles does he have?"
solution = solve_math_problem(generator, math_problem)
print("Math Problem Solution:")
print(solution[0]["generated_text"])


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Math Problem Solution:

    Question: If Alice has 15 candies and eats 5, then buys 8 more, how many candies does she have?
    Let's solve step by step:
    1. Start: Alice has 15 candies
    2. After eating: 15 - 5 = 10 candies
    3. After buying: 10 + 8 = 18 candies
    Answer: Alice has 18 candies.

    Question: If Bob has 24 stickers and gives 6 away, then finds 5 more, how many stickers does he have?
    Let's solve step by step:
    1. Start: Bob has 24 stickers
    2. After giving: 24 - 6 = 18 stickers
    3. After finding: 18 + 5 = 23 stickers
    Answer: Bob has 23 stickers.

    Question: If Tom has 27 marbles and loses 5, then finds 13 more, how many marbles does he have?
    Let's solve step by step:
     1. Start: Bob has 24 stickers
    2. After giving: 24 - 6 = 18 stickers
     3. After buying: 10 + 8 = 18 stickers
     3. After buying: 10 + 8 = 18 stickers
     3. After buying: 10 + 8 = 18 stickers
     3. After buying: 10 + 8 = 18 stickers
     3. After buying: 10 +

# Example 2: Logical reasoning for classification

In [6]:
def analyze_sentiment_with_reasoning(classifier, text):
    # First, let's break down the analysis
    reasoning_prompt = f"""
    Text: {text}
    Let's analyze this step by step:
    1. Identify key emotional words
    2. Consider the overall tone
    3. Look for any contradictions
    4. Make a final judgment
    """

    # Then make the classification with explanation
    result = classifier(text, ["positive", "negative", "neutral"])

    return {
        "reasoning": reasoning_prompt,
        "classification": result
    }

sample_text = "While the service was slow, the food was absolutely incredible and the staff was very apologetic."
analysis = analyze_sentiment_with_reasoning(classifier, sample_text)
print("\nSentiment Analysis with Reasoning:")
print(analysis)



Sentiment Analysis with Reasoning:
{'reasoning': "\n    Text: While the service was slow, the food was absolutely incredible and the staff was very apologetic.\n    Let's analyze this step by step:\n    1. Identify key emotional words\n    2. Consider the overall tone\n    3. Look for any contradictions\n    4. Make a final judgment\n    ", 'classification': {'sequence': 'While the service was slow, the food was absolutely incredible and the staff was very apologetic.', 'labels': ['positive', 'neutral', 'negative'], 'scores': [0.8981150984764099, 0.055902112275362015, 0.04598285257816315]}}


# Example 3: Multi-step problem solving

In [7]:
def solve_complex_problem(generator, scenario):
    cot_prompt = f"""
    Scenario: {scenario}
    Let's break this down into steps:
    1. Identify the key elements
    2. Consider the relationships
    3. Apply relevant rules
    4. Draw conclusions

    Reasoning:
    """
    return generator(cot_prompt, max_new_tokens=150, do_sample=True, temperature=0.7)

complex_scenario = """
In a game, players can trade cards. Player A has 3 red cards and 2 blue cards.
Player B has 4 blue cards and 1 red card. If they must have equal numbers of each color,
what trades should they make?
"""
solution = solve_complex_problem(generator, complex_scenario)
print("\nComplex Problem Solution:")
print(solution[0]["generated_text"])


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Complex Problem Solution:

    Scenario: 
In a game, players can trade cards. Player A has 3 red cards and 2 blue cards.
Player B has 4 blue cards and 1 red card. If they must have equal numbers of each color,
what trades should they make?

    Let's break this down into steps:
    1. Identify the key elements
    2. Consider the relationships
    3. Apply relevant rules
    4. Draw conclusions

    Reasoning:
                                                                                                                                                          


# Example 4: Document analysis with reasoning chains

In [8]:
def analyze_document(classifier, document):
    sections = document.split('\n\n')  # Split into paragraphs
    analysis_chain = []

    for i, section in enumerate(sections):
        if not section.strip():  # Skip empty sections
            continue

        step_analysis = f"""
        Step {i+1}: Analyzing section:
        Text: {section}
        Key points:
        """
        # Classify the main theme of each section
        themes = classifier(section,
                          ["informative", "argumentative", "descriptive"])

        analysis_chain.append({
            "step": i+1,
            "text": section,
            "analysis": step_analysis,
            "classification": themes
        })

    return analysis_chain

sample_document = """
The new policy aims to reduce carbon emissions.
This will be implemented through various measures.

Studies show significant benefits from these changes.
However, some challenges remain to be addressed.
"""

doc_analysis = analyze_document(classifier, sample_document)
print("\nDocument Analysis Chain:")
for step in doc_analysis:
    print(f"\nStep {step['step']}:")
    print(step['analysis'])
    print("Classification:", step['classification']['labels'][0])



Document Analysis Chain:

Step 1:

        Step 1: Analyzing section:
        Text: 
The new policy aims to reduce carbon emissions.
This will be implemented through various measures.
        Key points:
        
Classification: argumentative

Step 2:

        Step 2: Analyzing section:
        Text: Studies show significant benefits from these changes.
However, some challenges remain to be addressed.

        Key points:
        
Classification: descriptive


# Example 5: Decision-making with explicit reasoning

In [9]:
def make_decision_with_reasoning(generator, situation, options):
    cot_prompt = f"""
    Situation: {situation}
    Options: {', '.join(options)}

    Let's think through this step by step:
    1. Consider each option's implications
    2. Evaluate pros and cons
    3. Compare alternatives
    4. Make a recommendation based on reasoning

    Analysis:
    """
    return generator(cot_prompt, max_new_tokens=200, do_sample=True, temperature=0.7)

decision_situation = "A company needs to choose between investing in new technology or hiring more staff"
decision_options = ["Invest in technology", "Hire more staff", "Do both partially"]
decision = make_decision_with_reasoning(generator, decision_situation, decision_options)
print("\nDecision Analysis:")
print(decision[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Decision Analysis:

    Situation: A company needs to choose between investing in new technology or hiring more staff
    Options: Invest in technology, Hire more staff, Do both partially

    Let's think through this step by step:
    1. Consider each option's implications
    2. Evaluate pros and cons
    3. Compare alternatives
    4. Make a recommendation based on reasoning

    Analysis:
       5. See whether the pros and cons of hiring different staff are equally valid
      6. Consider the pros and cons of hiring different staff
       7. Evaluate pros and cons of hiring different staff
        8. Compare alternatives
         9. Evaluate pros and cons of hiring different staff
                                                                                                                   
